# 14 · Segmentación con K-Means

**Pregunta de clase.** ¿Hay perfiles parecidos en la encuesta simulada? K-Means agrupa personas por sus medidas. Aquí no damos al modelo una respuesta correcta que deba predecir.

## 0 · Preparar el entorno

Elige un kernel de **Python 3.11 o 3.12** con **Java 17**. En Jupyter o VS Code usa el mismo entorno que en las prácticas anteriores. En Colab, comprueba la versión con `!python --version`. Si muestra Python 3.13, en **Entorno de ejecución → Cambiar tipo de entorno de ejecución** selecciona la versión **2026.07**, que usa Python 3.12. Si no tienes Java 17, ejecuta en una celda de Colab `!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless` y, en otra celda, `import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"`.

Si faltan las bibliotecas, ejecuta en una celda `%pip install "pyspark>=3.5,<3.6" "numpy<2"` y reinicia el kernel si te lo pide. Puedes consultar la [guía del curso](../01_environment_pyspark.md). Ejecuta las celdas de arriba abajo. Los datos de ejemplo están dentro del notebook y el cálculo se hace en el equipo donde corre el kernel.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = (SparkSession.builder.master('local[2]').appName('Segmentacion K-Means')
         .config('spark.sql.shuffle.partitions', '4').getOrCreate())
print(f'Spark {spark.version}')

## 1 · Crear y revisar la tabla de ejemplo

Una fila representa a una persona ficticia. Las horas de formación previa y la jornada semanal se conocen antes de intentar predecir un resultado. El salario mensual y la finalización del curso son dos resultados diferentes. Usamos una semilla para generar la misma tabla en los cuadernos 11, 12 y 14.

La encuesta es **simulada**. Sirve para practicar el método y sus métricas.

In [ ]:
from random import Random

azar = Random(42)
filas = []

for persona in range(1, 361):
    # Tres perfiles de partida hacen visible la idea de segmentación.
    experiencia_base, jornada_base, formacion_base = azar.choice([
        (3, 24, 12), (8, 35, 28), (18, 40, 8),
    ])
    experiencia = max(0, experiencia_base + azar.randint(-3, 3))
    jornada = jornada_base + azar.randint(-4, 4)
    formacion = max(0, formacion_base + azar.randint(-8, 8))
    salario = round(650 + 60 * experiencia + 23 * jornada + azar.gauss(0, 220), 2)
    # La formación previa ayuda y una jornada más larga deja menos tiempo.
    puntuacion = formacion - 0.45 * jornada + azar.gauss(0, 9)
    completa = int(puntuacion >= 1)
    filas.append((persona, experiencia, jornada, formacion, salario, completa))

datos = spark.createDataFrame(filas, [
    'persona', 'anios_experiencia', 'horas_jornada', 'horas_formacion',
    'salario_mensual', 'completa_curso',
])

print(f'Filas: {datos.count()}')
datos.show(5)

## 2 · Elegir y escalar las medidas

Usamos experiencia, jornada y formación previa. Si una variable tiene valores mucho más dispersos, puede dominar la distancia. Dividimos cada medida por su desviación típica para que las escalas sean más comparables. Reservamos el salario y la finalización del curso para describir después los grupos.

In [ ]:
entradas = ['anios_experiencia', 'horas_jornada', 'horas_formacion']
vector = VectorAssembler(inputCols=entradas, outputCol='vector')
vectores = vector.transform(datos)
escalador = StandardScaler(inputCol='vector', outputCol='variables',
                            withStd=True, withMean=False).fit(vectores)
preparados = escalador.transform(vectores)
preparados.select('vector', 'variables').show(5, truncate=False)

## 3 · Probar dos y tres grupos

El parámetro `k` indica cuántos grupos buscamos. La silueta resume si las personas se parecen más a las de su grupo que a las de otros grupos. Un valor más alto indica mejor separación según esta medida. Aquí exploramos esta tabla y comparamos dos opciones con los mismos datos.

In [ ]:
evaluador = ClusteringEvaluator(featuresCol='variables', predictionCol='grupo')
resultados = []

for k in [2, 3]:
    modelo = KMeans(k=k, seed=42, featuresCol='variables', predictionCol='grupo').fit(preparados)
    predicciones = modelo.transform(preparados)
    silueta = evaluador.evaluate(predicciones)
    resultados.append((silueta, k, modelo))
    print(f'k = {k} · Silueta = {silueta:.3f}')

silueta, k_elegido, modelo_final = max(resultados, key=lambda resultado: resultado[0])
print(f'Opción elegida en esta comparación: k = {k_elegido}')

## 4 · Describir los perfiles en las unidades originales

Contamos las personas de cada grupo y volvemos a mostrar años, horas y euros. La tasa de finalización se calcula después de agrupar. El 0 y el 1 de la etiqueta permiten usar la media para obtener la proporción que terminó.

In [ ]:
perfiles = (modelo_final.transform(preparados).groupBy('grupo')
            .agg(F.count('*').alias('personas'),
                 F.round(F.avg('anios_experiencia'), 1).alias('experiencia_media'),
                 F.round(F.avg('horas_jornada'), 1).alias('jornada_media'),
                 F.round(F.avg('horas_formacion'), 1).alias('formacion_media'),
                 F.round(F.avg('salario_mensual'), 2).alias('salario_medio'),
                 F.round(100 * F.avg('completa_curso'), 1).alias('pct_completa')))
perfiles.orderBy('grupo').show()

**Interpretación.** Describe cada grupo por sus medidas. Por ejemplo, puedes observar si uno tiene más experiencia o más formación previa. Los grupos dependen de las variables, el escalado y el valor de k. La silueta ayuda a comparar estas opciones y la tabla ayuda a valorar si los grupos se entienden.

## Cerrar la sesión

Liberamos los recursos de Spark al terminar este cuaderno.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Volver al ejemplo de texto](13_spark_text_analysis.ipynb)

Documentación de Spark ML. [K-Means](https://spark.apache.org/docs/3.5.6/ml-clustering.html) · [Preparación de variables](https://spark.apache.org/docs/3.5.6/ml-features.html)